In [56]:
# CELL 1: Setup
# This notebook completes Phase 1 by generating a synthetic option chain
# and building the ATM lookup table for the backtest.

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
import warnings

warnings.filterwarnings("ignore")

# Add source modules to path
sys.path.append("../src")

from data_fetcher.synthetic_options import generate_synthetic_options
from preprocessing.atm_selector import build_atm_lookup

print("Environment ready.")

Environment ready.


In [57]:
# CELL 2: Load Configuration

config_path = Path("../config/config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded.")
print(f"Team: {config['project']['team']}")
print(f"Student ID: {config['project']['student_id']}")


Configuration loaded.
Team: Strike Squad
Student ID: 2024A1PS0271P


In [58]:
# CELL 3: Load Processed Stock and Risk-Free Data

processed_dir = Path("../data/processed")

stock_files = list(processed_dir.glob("hdfc_stock_*.csv"))
rf_files = list(processed_dir.glob("risk_free_rates_*.csv"))

if not stock_files or not rf_files:
    raise FileNotFoundError("Processed stock or risk-free data not found. Please run Notebook 01 first.")

stock_data = pd.read_csv(stock_files[0])
risk_free_data = pd.read_csv(rf_files[0])

stock_data["date"] = pd.to_datetime(stock_data["date"])
risk_free_data["date"] = pd.to_datetime(risk_free_data["date"])

print(f"Stock rows loaded      : {len(stock_data)}")
print(f"Risk-free rows loaded  : {len(risk_free_data)}")
print(f"Date range             : {stock_data['date'].min().strftime('%Y-%m-%d')} → {stock_data['date'].max().strftime('%Y-%m-%d')}")


Stock rows loaded      : 508
Risk-free rows loaded  : 752
Date range             : 2023-11-01 → 2025-11-20


In [59]:
# CELL 4: Generate Synthetic Option Chain
# Synthetic option prices are generated using the Black–Scholes model.
# This approach is used because historical NSE option chain data is not freely accessible.

print("Generating synthetic option chain. This may take a few minutes...\n")

# Normalize datetime columns before generating options
stock_data["date"] = pd.to_datetime(stock_data["date"]).dt.tz_localize(None)
risk_free_data["date"] = pd.to_datetime(risk_free_data["date"]).dt.tz_localize(None)

option_chain = generate_synthetic_options(
    stock_df=stock_data,
    rf_df=risk_free_data,
    config=config
)

print(f"Option chain generated: {len(option_chain):,} rows.")
print("\nPreview:")
print(option_chain.head())


Generating synthetic option chain. This may take a few minutes...

Option chain generated: 8,694 rows.

Preview:
        date     expiry  strike  days_to_expiry  stock_price  call_bid  \
0 2023-12-15 2024-01-18     720              34   806.219299     90.35   
1 2023-12-15 2024-01-18     730              34   806.219299     80.45   
2 2023-12-15 2024-01-18     740              34   806.219299     70.65   
3 2023-12-15 2024-01-18     750              34   806.219299     61.00   
4 2023-12-15 2024-01-18     760              34   806.219299     51.50   

   call_ask   call_mid  put_bid  put_ask   put_mid  implied_vol  \
0     91.25  90.781899     0.05     0.00  0.016356     0.134515   
1     81.30  80.878702     0.05     0.05  0.050017     0.134515   
2     71.40  71.028832     0.15     0.15  0.137004     0.134515   
3     61.60  61.293289     0.35     0.35  0.338319     0.134515   
4     52.05  51.776161     0.75     0.75  0.758049     0.134515   

   open_interest  volume  risk_free_rat

In [60]:
# CELL 5: Validate Synthetic Option Chain

unique_dates = option_chain["date"].nunique()
avg_strikes = option_chain.groupby("date")["strike"].nunique().mean()

print(f"Trading days covered       : {unique_dates}")
print(f"Average strikes per day    : {avg_strikes:.1f}")

call_stats = option_chain["call_mid"].describe()
put_stats = option_chain["put_mid"].describe()

print("\nCall Price Summary:")
print(call_stats)

print("\nPut Price Summary:")
print(put_stats)


Trading days covered       : 478
Average strikes per day    : 18.2

Call Price Summary:
count    8694.000000
mean       34.639221
std        31.153229
min         0.005383
25%         6.826871
50%        25.013650
75%        58.694596
max       115.753232
Name: call_mid, dtype: float64

Put Price Summary:
count    8694.000000
mean       24.381344
std        25.159650
min         0.000014
25%         2.565373
50%        14.367643
75%        42.376415
max        94.108810
Name: put_mid, dtype: float64


In [61]:
# CELL 6: Build ATM Lookup Table

atm_lookup = build_atm_lookup(
    stock_data,
    option_chain,
    config
)

print(f"ATM lookup entries created: {len(atm_lookup)}")
print("\nPreview:")
atm_lookup.head()


ATM lookup entries created: 478

Preview:


,trade_date,expiry_date,days_to_expiry,stock_price,atm_strike,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,moneyness,implied_vol,open_interest,volume,risk_free_rate,call_spread_pct,put_spread_pct,liquid,skip_reason
0,2023-12-15,2024-01-18,34,806.219299,810,13.75,13.95,13.870425,12.45,12.65,12.536601,0.995332,0.134515,5948,150,0.068,1.441917,1.595329,True,
1,2023-12-18,2024-01-18,31,805.805542,810,12.80,12.95,12.875233,12.30,12.50,12.405147,0.994822,0.134954,5110,384,0.068,1.165027,1.612234,True,
2,2023-12-19,2024-01-18,30,804.442871,800,17.25,17.50,17.384244,8.35,8.60,8.482612,1.005554,0.135855,5910,506,0.068,1.438084,2.947206,True,
3,2023-12-20,2024-01-25,36,806.438232,810,14.50,14.70,14.615347,12.65,12.85,12.762743,0.995603,0.135452,5290,500,0.068,1.368425,1.567061,True,
4,2023-12-21,2024-01-25,35,820.892822,820,17.25,17.50,17.393614,11.10,11.25,11.171337,1.001089,0.139222,5763,246,0.068,1.437309,1.342722,True,


In [62]:
# CELL 7: ATM Filtering Summary

liquid_count = atm_lookup["liquid"].sum()
total_count = len(atm_lookup)
skip_count = total_count - liquid_count

print(f"Total potential trades : {total_count}")
print(f"Liquid trades          : {liquid_count} ({liquid_count/total_count:.2%})")
print(f"Skipped trades         : {skip_count} ({skip_count/total_count:.2%})")

m_stats = atm_lookup["moneyness"].describe()

print("\nMoneyness Summary:")
print(m_stats)


Total potential trades : 478
Liquid trades          : 478 (100.00%)
Skipped trades         : 0 (0.00%)

Moneyness Summary:
count    478.000000
mean       1.000176
std        0.003491
min        0.993479
25%        0.997079
50%        1.000313
75%        1.003154
max        1.007087
Name: moneyness, dtype: float64


In [63]:
# CELL 8: Save Phase 1 Outputs

output_dir = Path("../data/processed")

option_chain_file = output_dir / "option_chain_synthetic_0271.csv"
option_chain.to_csv(option_chain_file, index=False)

atm_file = output_dir / "atm_lookup_table_0271.csv"
atm_lookup.to_csv(atm_file, index=False)

liquid_only = atm_lookup[atm_lookup["liquid"]]
liquid_file = output_dir / "atm_lookup_liquid_only_0271.csv"
liquid_only.to_csv(liquid_file, index=False)

print("Files saved:")
print(f" - {option_chain_file.name}")
print(f" - {atm_file.name}")
print(f" - {liquid_file.name}")


Files saved:
 - option_chain_synthetic_0271.csv
 - atm_lookup_table_0271.csv
 - atm_lookup_liquid_only_0271.csv


In [64]:
# CELL 9: Phase 1 Completion Summary

print("Phase 1 Summary\n")

print(f"Stock data rows           : {len(stock_data)}")
print(f"Option chain rows         : {len(option_chain):,}")
print(f"ATM lookup entries        : {len(atm_lookup)}")
print(f"Liquid trades             : {len(liquid_only)} ({len(liquid_only)/len(atm_lookup):.2%})")

print("\nPhase 1 is complete. The data is now ready for Phase 2:\n"
      "- Black-Scholes model implementation\n"
      "- Binomial model with multiple step sizes\n"
      "- Convergence analysis")

print("\nProceed to Notebook 02 (Pricing Models).")


Phase 1 Summary

Stock data rows           : 508
Option chain rows         : 8,694
ATM lookup entries        : 478
Liquid trades             : 478 (100.00%)

Phase 1 is complete. The data is now ready for Phase 2:
- Black-Scholes model implementation
- Binomial model with multiple step sizes
- Convergence analysis

Proceed to Notebook 02 (Pricing Models).


In [65]:
# FINAL CELL — Save synthetic option chain

from pathlib import Path

output_path = Path("../data/raw/hdfc_option_chain_raw.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

if "option_chain" not in globals() or option_chain is None:
    raise ValueError("option_chain DataFrame not found. Run generation cell first.")

option_chain.to_csv(output_path, index=False)

print("Synthetic option chain successfully saved to:")
print(output_path.resolve())
print("Total rows:", len(option_chain))


Synthetic option chain successfully saved to:
/Users/shubhra/hdfcbank-strike-squad/data/raw/hdfc_option_chain_raw.csv
Total rows: 8694


In [66]:
option_chain.head()


,date,expiry,strike,days_to_expiry,stock_price,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,risk_free_rate,moneyness,call_spread_pct,put_spread_pct
0,2023-12-15,2024-01-18,720,34,806.219299,90.35,91.25,90.781899,0.05,0.00,0.016356,0.134515,4533,247,0.068,1.119749,0.991387,-305.701209
1,2023-12-15,2024-01-18,730,34,806.219299,80.45,81.30,80.878702,0.05,0.05,0.050017,0.134515,3876,360,0.068,1.104410,1.050957,0.000000
2,2023-12-15,2024-01-18,740,34,806.219299,70.65,71.40,71.028832,0.15,0.15,0.137004,0.134515,4376,120,0.068,1.089486,1.055909,0.000000
3,2023-12-15,2024-01-18,750,34,806.219299,61.00,61.60,61.293289,0.35,0.35,0.338319,0.134515,5646,507,0.068,1.074959,0.978900,0.000000
4,2023-12-15,2024-01-18,760,34,806.219299,51.50,52.05,51.776161,0.75,0.75,0.758049,0.134515,5792,457,0.068,1.060815,1.062265,0.000000


In [67]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Date range
start = datetime(2023, 1, 2)
end = datetime(2025, 11, 21)
dates = pd.date_range(start, end, freq="D")

# Base rate assumptions (smooth realistic RBI T-Bill trend)
# 2023 → 6.45%
# 2024 → 6.60%
# 2025 → 6.70%
base_2023 = 0.0645
base_2024 = 0.0660
base_2025 = 0.0670

rates = []

for d in dates:
    if d.year == 2023:
        # slow upward drift
        t = (d - datetime(2023,1,2)).days
        rate = base_2023 + (t * 0.0000015)
        
    elif d.year == 2024:
        t = (d - datetime(2024,1,1)).days
        rate = base_2024 + (t * 0.0000012)
        
    else:  # 2025
        t = (d - datetime(2025,1,1)).days
        rate = base_2025 + (t * 0.0000010)
    
    rates.append(rate)

df = pd.DataFrame({
    "date": dates,
    "rate": rates
})

# Save file
output_path = "../data/raw/risk_free_raw.csv"
df.to_csv(output_path, index=False)

print(f"Risk-free file created at: {output_path}")
print(df.head())
print(df.tail())


Risk-free file created at: ../data/raw/risk_free_raw.csv
        date      rate
0 2023-01-02  0.064500
1 2023-01-03  0.064502
2 2023-01-04  0.064503
3 2023-01-05  0.064505
4 2023-01-06  0.064506
           date      rate
1050 2025-11-17  0.067320
1051 2025-11-18  0.067321
1052 2025-11-19  0.067322
1053 2025-11-20  0.067323
1054 2025-11-21  0.067324


In [68]:
import pandas as pd
import yfinance as yf
from pathlib import Path

# Ticker for HDFC Bank (NSE)
ticker = yf.Ticker("HDFCBANK.NS")

print("Fetching dividend history from Yahoo Finance...\n")

# Fetch dividends (returns a Pandas Series: index=dates, values=amount)
div_series = ticker.dividends

if div_series.empty:
    raise ValueError("No dividend data could be fetched for HDFCBANK.NS")

# Convert to DataFrame
div_df = div_series.reset_index()
div_df.columns = ["Ex_Date", "Dividend_Amount"]

# Sort by date
div_df = div_df.sort_values("Ex_Date").reset_index(drop=True)

# Filter to reasonable range (your project window)
div_df = div_df[
    (div_df["Ex_Date"] >= "2022-01-01") &
    (div_df["Ex_Date"] <= "2025-12-31")
]

# Output path
output_path = Path("../data/raw/hdfc_dividends_raw.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save CSV
div_df.to_csv(output_path, index=False)

print("Dividend file generated successfully!")
print(f"Saved at: {output_path}\n")
print("Preview:")
display(div_df.head(10))
display(div_df.tail(10))


Fetching dividend history from Yahoo Finance...

Dividend file generated successfully!
Saved at: ../data/raw/hdfc_dividends_raw.csv

Preview:


,Ex_Date,Dividend_Amount
25,2022-05-12 00:00:00+05:30,7.75
26,2023-05-16 00:00:00+05:30,9.50
27,2024-05-10 00:00:00+05:30,9.75
28,2025-06-27 00:00:00+05:30,11.00
29,2025-07-25 00:00:00+05:30,2.50


,Ex_Date,Dividend_Amount
25,2022-05-12 00:00:00+05:30,7.75
26,2023-05-16 00:00:00+05:30,9.50
27,2024-05-10 00:00:00+05:30,9.75
28,2025-06-27 00:00:00+05:30,11.00
29,2025-07-25 00:00:00+05:30,2.50


In [3]:
# === Universal PDF Export Cell (Jupyter, macOS, LaTeX-free, automatic install) ===
import os, sys, subprocess, importlib
from nbconvert import HTMLExporter

# ---- Step 1: Install Playwright if missing ----
def ensure_playwright():
    spec = importlib.util.find_spec("playwright")
    if spec is None:
        print("Playwright not found. Installing into this kernel environment...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "playwright", "nbconvert"], check=True)
        print("Installing Chromium browser (required for PDF generation)...")
        subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
        print("Playwright + Chromium installed.")
    else:
        print("Playwright already installed.")

ensure_playwright()

# After install, import async API
from playwright.async_api import async_playwright

# ---- Step 2: Ask for notebook filename ----
notebook_name = input("Enter notebook name (e.g. 01_data_acquisition.ipynb): ").strip()
if not notebook_name:
    raise SystemExit("Notebook name required.")
if not os.path.exists(notebook_name):
    raise FileNotFoundError(f"Notebook not found: {notebook_name}")

html_name = notebook_name.replace(".ipynb", ".html")
pdf_name = notebook_name.replace(".ipynb", ".pdf")

# ---- Step 3: Export notebook → HTML ----
print(f"Exporting {notebook_name} → {html_name} ...")
html_exporter = HTMLExporter()
html_exporter.exclude_input = False
html_exporter.template_name = "classic"

body, resources = html_exporter.from_filename(notebook_name)
with open(html_name, "w", encoding="utf-8") as f:
    f.write(body)
print(f"HTML saved as {html_name}")

# ---- Step 4: Render HTML → PDF using Chromium headless ----
abs_html = os.path.abspath(html_name)
file_url = "file://" + abs_html
print("Rendering PDF using Chromium...")

async def _export_pdf():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        ctx = await browser.new_context()
        page = await ctx.new_page()
        await page.goto(file_url, wait_until="networkidle")
        await page.pdf(
            path=pdf_name,
            format="A4",
            print_background=True,
            margin={"top": "12mm", "bottom": "12mm", "left": "12mm", "right": "12mm"}
        )
        await browser.close()

await _export_pdf()

print(f"PDF successfully created: {pdf_name}")


Playwright not found. Installing into this kernel environment...
Installing Chromium browser (required for PDF generation)...
Playwright + Chromium installed.


Enter notebook name (e.g. 01_data_acquisition.ipynb):  01B_option_chain_generation.ipynb


Exporting 01B_option_chain_generation.ipynb → 01B_option_chain_generation.html ...
HTML saved as 01B_option_chain_generation.html
Rendering PDF using Chromium...
PDF successfully created: 01B_option_chain_generation.pdf
